In [4]:
import os
import json

data = []
for i in range(1, 11):
    file_path = f"/home/mhtuan/work/mbf/result/1week_data.txt_part_{i}.json"
    with open(file_path, "r") as f:
        lines = f.readlines()
        for l in lines:
            json_line = json.loads(l.strip())
            if len(json_line['event_triggers']) > 0 and len(json_line['entity_mentions']) > 0:
                data.append(json_line)

with open("/home/mhtuan/work/mbf/result/filtered.json", "w+") as f:
    for d in data:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [ ]:
import os
import json

data = []
for i in range(1, 11):
    file_path = f"/home/mhtuan/work/mbf/result/1week_data.txt_part_{i}.json"
    with open(file_path, "r") as f:
        lines = f.readlines()
        for l in lines:
            json_line = json.loads(l.strip())
            if len(json_line['event_triggers']) > 0 and len(json_line['entity_mentions']) > 0:
                data.append(json_line)

with open("/home/mhtuan/work/mbf/result/filtered.json", "w+") as f:
    for d in data:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [3]:
import re
from urllib.parse import urlparse

anh_pattern = r"\(\s*Ảnh\s*\s*.*?\)"

def extract_path_part(url):
    parsed_url = urlparse(url)
    path = parsed_url.path.strip("/")
    match = re.search(r"([^/]+)", path)
    return match.group(1) if match else None

def add_spaces(text):
    return ' '.join(re.findall(r'\w+|[^\w\s]', text, re.UNICODE))

def count_words(text):
    words = re.findall(r'\w+', text, re.UNICODE)
    return len(words)

def format_num(text: str):
    result = re.sub(r"(?<=[A-ZĐ])\.(?=[A-ZĐ])", "", text)
    result = re.sub(r"(?<=\d)\.(?=\d)", "", result)
    return result

def normalize_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def sentence_segment(text: str) -> set:
    ttt = text.replace("...", "").replace(".)", ")").replace("'s", "")
    if ttt == "":
        return set()
    ttt = normalize_spaces(ttt)
    sentences = re.split(r'(?<=[.?!])\s?', format_num(ttt))
    
    formatted_sentences = []
    for s in sentences:
        if s.strip() == "" or count_words(s.strip()) < 7:
            continue
        f_s = add_spaces(s).replace("”", '"').replace("“", '"')
        f_s = re.sub(anh_pattern, "", f_s).strip()

        formatted_sentences.append(f_s)

    return set(formatted_sentences)


# for index, cate in enumerate(result):
#     file_name = extract_path_part(cate['cate'])
#     for record in cate['data']:
#         ttt = record['text'].replace("...", "").replace(".)", ")").replace("'s", "")
#         if ttt == "":
#             continue
#         ttt = normalize_spaces(ttt)
#         sentences = re.split(r'(?<=[.?!])\s?', format_num(ttt))
        
#         formatted_sentences = []
#         for s in sentences:
#             if s.strip() == "" or count_words(s.strip()) < 7:
#                 continue
#             f_s = add_spaces(s).replace("”", '"').replace("“", '"')
#             f_s = re.sub(anh_pattern, "", f_s).strip()

#             formatted_sentences.append(f_s)
#             with open(f"/home/mhtuan/work/fourie/testing/crawled/ggnews_{index}.txt", "a+") as ff:
#                 ff.write(normalize_spaces(f_s) + "\n")
        # print(formatted_sentences)
        # record['sentences'] = formatted_sentences
        # break

# with open("by_topics_filtered.json", "w+", encoding='utf-8') as ff:
#     json.dump(result, ff, ensure_ascii=False, indent=4)

In [17]:
formatted_sentences = []
with open('postgres_public_news_urls.txt', 'r') as f:
    sentences = f.readlines()
    for s in sentences:
        s = s.replace(".", "").strip()
        if s.strip() == "" or count_words(s.strip()) < 7:
            continue
        f_s = add_spaces(s).replace("”", '"').replace("“", '"')
        f_s = re.sub(anh_pattern, "", f_s).strip()

        formatted_sentences.append(f_s)

with open('postgres_public_news_urls.txt', 'w+') as f:
    for s in formatted_sentences:
        f.write(s + "\n")

In [22]:
import json

with open("/home/mhtuan/work/mbf/Result_22.json") as f:
    title_mapping = json.load(f)

# for item in title_mapping:
#     s = item['title'].replace(".", "").strip()
#     if s.strip() == "" or count_words(s.strip()) < 7:
#         continue
#     f_s = add_spaces(s).replace("”", '"').replace("“", '"')
#     f_s = re.sub(anh_pattern, "", f_s).strip()

#     item['title'] = f_s

# with open("/home/mhtuan/work/mbf/Result_22.json", "w+") as f:
#     f.write(json.dumps(title_mapping, ensure_ascii=False))

In [23]:
predicted_data = []

with open('postgres_public_news_urls.json', 'r') as f:
    lines = f.readlines()
    for l in lines:
        json_line = json.loads(l.strip())
        if len(json_line['event_triggers']) == 0 or len(json_line['entity_mentions']) == 0:
            continue
        
        matched_index = -1
        for index, item in enumerate(title_mapping):
            if json_line['text'].replace('"', '').strip() != item['title']:
                continue
            json_line['url'] = item['url']
            json_line['date'] = item['date']
            matched_index = index
            break
        
        if matched_index == -1:
            print(json_line['text'].replace('"', '').strip())
            raise Exception("sai dcmm")
        
        title_mapping.pop(matched_index)
        predicted_data.append(json_line)

In [24]:
with open('postgres_public_news_urls.json', 'w+') as f:
    for item in predicted_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [6]:
import psycopg2

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",  # e.g., "localhost" or an IP address
    port="5432"  # Default PostgreSQL port
)

cur = conn.cursor()

# Example query
cur.execute("""
SELECT * FROM news_urls
WHERE text like '%Sae%' AND text like '%hẹn hò%'AND
    published_time >= '2025-03-07'
""")
rows = cur.fetchall()

# Close the cursor and connection
cur.close()
conn.close()

with open("kim-soo-huyn.txt", "w+") as f:
    for r in rows:
        sentences = sentence_segment(text=r[-1])
        for s in sentences:
            f.write(s + "\n")

In [4]:
import json

data = []
file_path = f"/home/mhtuan/work/mbf/result/filtered_matched_url.json"
with open(file_path, "r") as f:
    lines = f.readlines()
    for l in lines:
        json_line = json.loads(l.strip())
        json_line['date'] = str(dict_date[json_line['url']])
        # for r in dict_raw.keys():
        #     if json_line['text'] in dict_raw[r]:
        #         json_line['url'] = r
        #         break
        # if "url" in json_line:
        data.append(json_line)

with open("/home/mhtuan/work/mbf/result/filtered_matched_date.json", "w+") as f:
    for d in data:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [11]:
file_path = f"/home/mhtuan/work/mbf/result/filtered_matched_url.json"
with open(file_path, "r") as f:
    lines = f.readlines()
    for l in lines:
        json_line = json.loads(l.strip())
        if json_line['url'] == None:
            # print(json_line)
            print(f"SELECT * FROM news_urls WHERE text LIKE '%{json_line['text']}%';")

SELECT * FROM news_urls WHERE text LIKE '%Một chuyến trao tặng phòng Tin học .%';
SELECT * FROM news_urls WHERE text LIKE '%Hơn 2000 chiếc máy tính đã được trao tặng sau gần 2 năm .%';
SELECT * FROM news_urls WHERE text LIKE '%Ông Trump đã mua một chiếc Tesla Model S màu đỏ và mua xe Tesla Cybertruck tặng cháu gái .%';
SELECT * FROM news_urls WHERE text LIKE '%Người thân của các nạn nhân trong cuộc chiến chống ma túy đẫm máu của cựu Tổng thống Philippines Rodrigo Duterte khóc trong lễ cầu nguyện cho các nạn nhân tại một nhà thờ ở Manila , sau khi ông bị bắt hôm thứ Ba .%';
SELECT * FROM news_urls WHERE text LIKE '%Từ trái sang phải : Cố vấn An ninh Quốc gia Mỹ Mike Waltz , Ngoại trưởng Mỹ Marco Rubio , Bộ trưởng Ngoại giao Ả Rập Xê Út Faisal bin Farhan , Cố vấn An ninh Quốc gia Ả Rập Xê Út Mosaad bin Mohammad al - Aiban , Bộ trưởng Ngoại giao Ukraine Andrii Sybiha và Chánh Văn phòng Tổng thống Ukraine Andriy Yermak họp tại Jeddah , Ả Rập Xê Út .%';
SELECT * FROM news_urls WHERE text LI

In [1]:
import requests, time, psycopg2, json, traceback
from dateutil import parser
from bs4 import BeautifulSoup

DB_PARAMS = {
    "dbname": "postgres",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",  # e.g., "localhost" or an IP address
    "port": "5432",  # Default PostgreSQL port
}
conn = psycopg2.connect(**DB_PARAMS)

def access(url: str):
    while (True):
        try:
            res = requests.get(url, timeout=5)
            return res.text
        except Exception as e:
            time.sleep(1)
            continue

def extract_content(url: str, cur, conn):
    print(url)
    try:
        soup = BeautifulSoup(access(url=url), 'html.parser')
        script = soup.select_one("#__NEXT_DATA__")
        script_json = json.loads(script.text)

        texts = [t['content'] for t in script_json['props']['pageProps']['resp']['data']['content']['bodys'] if t['type'] == 'text']
        full_text = ""
        for t in texts:
            pt = t
            if len(t) == 0:
                continue
            if t[-1] != ".":
                pt += "."
            # print(t)
            full_text += (BeautifulSoup(pt, "html.parser").text.replace("\n", " ").strip() + " ")

        published_time = script_json['props']['pageProps']['resp']['data']['head']['meta']['articlePublishedTime']
        dt = parser.isoparse(published_time).replace(tzinfo=None)
        # print(dt)

        insert_query = """INSERT INTO news_urls (url, published_time, text) VALUES (%s, %s, %s)
        ON CONFLICT (url) DO UPDATE SET
        url = EXCLUDED.url,
        published_time = EXCLUDED.published_time,
        text = EXCLUDED.text
        """
        cur.execute(insert_query, (url, dt, full_text))
        conn.commit()
    except Exception as e:
        print(script)
        print(traceback.format_exc())

urls = ["https://baomoi.com/kim-soo-hyun-bat-ngo-len-tieng-vu-be-boi-thua-nhan-quan-he-voi-kim-sae-ron-c51711795.epi",
"https://baomoi.com/tuyen-bo-gay-soc-cua-kim-soo-hyun-ve-chuyen-hen-ho-voi-kim-sae-ron-c51710404.epi",
"https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-lo-dau-hieu-bat-on-tam-ly-c51710575.epi",
"https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-c51710647.epi",
"https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-hien-dang-bat-on-tam-ly-c51710937.epi",
"https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-tu-nam-2019-da-qua-tuoi-vi-thanh-nien-c51711548.epi",
"https://baomoi.com/kim-soo-hyun-bat-ngo-quay-xe-thua-nhan-hen-ho-kim-sae-ron-c51711144.epi",
"https://baomoi.com/kim-soo-hyun-bat-ngo-quay-xe-thua-nhan-tung-hen-ho-kim-sae-ron-c51711496.epi"]

cur = conn.cursor()
for url in urls:
    extract_content(url=url, cur=cur, conn=conn)

https://baomoi.com/kim-soo-hyun-bat-ngo-len-tieng-vu-be-boi-thua-nhan-quan-he-voi-kim-sae-ron-c51711795.epi
https://baomoi.com/tuyen-bo-gay-soc-cua-kim-soo-hyun-ve-chuyen-hen-ho-voi-kim-sae-ron-c51710404.epi
https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-lo-dau-hieu-bat-on-tam-ly-c51710575.epi
https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-c51710647.epi
https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-hien-dang-bat-on-tam-ly-c51710937.epi
https://baomoi.com/kim-soo-hyun-thua-nhan-hen-ho-kim-sae-ron-tu-nam-2019-da-qua-tuoi-vi-thanh-nien-c51711548.epi


/tmp/ipykernel_39971/2065016590.py:39: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  full_text += (BeautifulSoup(pt, "html.parser").text.replace("\n", " ").strip() + " ")


https://baomoi.com/kim-soo-hyun-bat-ngo-quay-xe-thua-nhan-hen-ho-kim-sae-ron-c51711144.epi
https://baomoi.com/kim-soo-hyun-bat-ngo-quay-xe-thua-nhan-tung-hen-ho-kim-sae-ron-c51711496.epi


In [26]:
from bs4 import BeautifulSoup

html = """<div class="list content-list group/list"><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/hot-girl-bac-ninh-gay-sot-nhung-ngay-qua-vi-drama-cam-sung-c51782952.epi" class="" title="Hot girl Bắc Ninh gây sốt những ngày qua vì drama cắm sừng" target="_blank" rel="noopener noreferrer"><figure class="image overflow-hidden cursor-pointer"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_23_180_51782952/06cdaa25aa6b43351a7a.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_23_180_51782952/06cdaa25aa6b43351a7a.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_23_180_51782952/06cdaa25aa6b43351a7a.jpg" alt="Hot girl Bắc Ninh gây sốt những ngày qua vì drama cắm sừng"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/hot-girl-bac-ninh-gay-sot-nhung-ngay-qua-vi-drama-cam-sung-c51782952.epi" class="" title="Hot girl Bắc Ninh gây sốt những ngày qua vì drama cắm sừng" target="_blank" rel="noopener noreferrer">Hot girl Bắc Ninh gây sốt những ngày qua vì drama cắm sừng</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tri-thuc-cuoc-song-tri-thuc-cuoc-song-p180.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tri thức &amp; Cuộc sống"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:64px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" alt="Báo Tri thức &amp; Cuộc sống" width="256" height="64"></picture></figure></a><time class="content-time empty:hidden" datetime="2025-03-23T08:30:00+07:00">31 phút</time><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/hot-girl-bac-ninh-gay-sot-nhung-ngay-qua-vi-drama-cam-sung-c51782952.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Hot girl Bắc Ninh gây sốt những ngày qua vì drama cắm sừng"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-750px] group-hover/card:before:bg-[position:-50px_-750px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/phi-phuong-anh-len-tieng-giua-on-ao-cua-viruss-gio-thi-tin-chua-c51781040.epi" class="" title="Phí Phương Anh lên tiếng giữa ồn ào của ViruSs: 'Giờ thì tin chưa'" target="_blank" rel="noopener noreferrer"><figure class="image overflow-hidden cursor-pointer"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_329_51781040/c34707d5009be9c5b08a.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_329_51781040/c34707d5009be9c5b08a.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_329_51781040/c34707d5009be9c5b08a.jpg" alt="Phí Phương Anh lên tiếng giữa ồn ào của ViruSs: 'Giờ thì tin chưa'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/phi-phuong-anh-len-tieng-giua-on-ao-cua-viruss-gio-thi-tin-chua-c51781040.epi" class="" title="Phí Phương Anh lên tiếng giữa ồn ào của ViruSs: 'Giờ thì tin chưa'" target="_blank" rel="noopener noreferrer">Phí Phương Anh lên tiếng giữa ồn ào của ViruSs: 'Giờ thì tin chưa'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-saostar-saostar-p329.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí SaoStar"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:54px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" alt="Tạp chí SaoStar" width="216" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/phi-phuong-anh-len-tieng-giua-on-ao-cua-viruss-gio-thi-tin-chua-c51781040.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Phí Phương Anh lên tiếng giữa ồn ào của ViruSs: 'Giờ thì tin chưa'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/soc-voi-phao-c51780651.epi" class="" title="Sốc với Pháo" target="_blank" rel="noopener noreferrer"><figure class="image overflow-hidden cursor-pointer"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51780651/4793b216b5585c060549.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51780651/4793b216b5585c060549.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51780651/4793b216b5585c060549.jpg" alt="Sốc với Pháo"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/soc-voi-phao-c51780651.epi" class="" title="Sốc với Pháo" target="_blank" rel="noopener noreferrer">Sốc với Pháo</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tien-phong-tien-phong-p20.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:50px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" alt="Báo Tiền Phong" width="200" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/soc-voi-phao-c51780651.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Sốc với Pháo"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-la-chu-tich-giam-doc-nhieu-doanh-nghiep-c51779986.epi" class="" title="ViruSs là chủ tịch, giám đốc nhiều doanh nghiệp" target="_blank" rel="noopener noreferrer"><figure class="image overflow-hidden cursor-pointer"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51779986/df604ec64988a0d6f999.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51779986/df604ec64988a0d6f999.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51779986/df604ec64988a0d6f999.jpg" alt="ViruSs là chủ tịch, giám đốc nhiều doanh nghiệp"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-la-chu-tich-giam-doc-nhieu-doanh-nghiep-c51779986.epi" class="" title="ViruSs là chủ tịch, giám đốc nhiều doanh nghiệp" target="_blank" rel="noopener noreferrer">ViruSs là chủ tịch, giám đốc nhiều doanh nghiệp</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/viruss-la-chu-tich-giam-doc-nhieu-doanh-nghiep-c51779986.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs là chủ tịch, giám đốc nhiều doanh nghiệp"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-va-ngoc-kem-dau-to-phao-len-tieng-phu-nhan-tin-don-tinh-cam-ra-bai-rap-diss-cuc-ben-c51779581.epi" class="" title="ViruSs và Ngọc Kem đấu tố, Pháo lên tiếng phủ nhận tin đồn tình cảm, ra bài rap diss cực 'bén'" target="_blank" rel="noopener noreferrer"><figure class="image overflow-hidden cursor-pointer"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_304_51779581/b7088adc8d9264cc3d83.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_304_51779581/b7088adc8d9264cc3d83.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_304_51779581/b7088adc8d9264cc3d83.jpg" alt="ViruSs và Ngọc Kem đấu tố, Pháo lên tiếng phủ nhận tin đồn tình cảm, ra bài rap diss cực 'bén'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-va-ngoc-kem-dau-to-phao-len-tieng-phu-nhan-tin-don-tinh-cam-ra-bai-rap-diss-cuc-ben-c51779581.epi" class="" title="ViruSs và Ngọc Kem đấu tố, Pháo lên tiếng phủ nhận tin đồn tình cảm, ra bài rap diss cực 'bén'" target="_blank" rel="noopener noreferrer">ViruSs và Ngọc Kem đấu tố, Pháo lên tiếng phủ nhận tin đồn tình cảm, ra bài rap diss cực 'bén'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-doanh-nghiep-viet-nam-doanh-nghiep-p304.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Doanh Nghiệp Việt Nam"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:46.03508771929825px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/62680803114ef810a15f.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/62680803114ef810a15f.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/62680803114ef810a15f.png" alt="Tạp chí Doanh Nghiệp Việt Nam" width="164" height="57"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/viruss-va-ngoc-kem-dau-to-phao-len-tieng-phu-nhan-tin-don-tinh-cam-ra-bai-rap-diss-cuc-ben-c51779581.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs và Ngọc Kem đấu tố, Pháo lên tiếng phủ nhận tin đồn tình cảm, ra bài rap diss cực 'bén'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-750px] group-hover/card:before:bg-[position:-50px_-750px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/ho-so-tinh-ai-cua-streamer-viruss-lan-nao-chia-tay-cung-on-ao-khap-coi-mang-c51777181.epi" class="" title="Hồ sơ tình ái của streamer ViruSs: Lần nào chia tay cũng ồn ào khắp cõi mạng!" target="_blank" rel="noopener noreferrer"><video playsinline=""><source src="https://gif-baomoi.bmcdn.me/w250_r3x2/2025_03_22_105_51777181/e86b706676289f76c639.gif.webm" type="video/webm"><source src="https://gif-baomoi.bmcdn.me/w250_r3x2/2025_03_22_105_51777181/e86b706676289f76c639.gif.mp4" type="video/mp4"></video></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ho-so-tinh-ai-cua-streamer-viruss-lan-nao-chia-tay-cung-on-ao-khap-coi-mang-c51777181.epi" class="" title="Hồ sơ tình ái của streamer ViruSs: Lần nào chia tay cũng ồn ào khắp cõi mạng!" target="_blank" rel="noopener noreferrer">Hồ sơ tình ái của streamer ViruSs: Lần nào chia tay cũng ồn ào khắp cõi mạng!</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:27px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/ho-so-tinh-ai-cua-streamer-viruss-lan-nao-chia-tay-cung-on-ao-khap-coi-mang-c51777181.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Hồ sơ tình ái của streamer ViruSs: Lần nào chia tay cũng ồn ào khắp cõi mạng!"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/phao-lam-bung-no-mang-xa-hoi-c51776725.epi" class="" title="Pháo làm bùng nổ mạng xã hội" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51776725/cd73635c65128c4cd503.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51776725/cd73635c65128c4cd503.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_20_51776725/cd73635c65128c4cd503.jpg" alt="Pháo làm bùng nổ mạng xã hội"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/phao-lam-bung-no-mang-xa-hoi-c51776725.epi" class="" title="Pháo làm bùng nổ mạng xã hội" target="_blank" rel="noopener noreferrer">Pháo làm bùng nổ mạng xã hội</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tien-phong-tien-phong-p20.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:50px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" alt="Báo Tiền Phong" width="200" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/phao-lam-bung-no-mang-xa-hoi-c51776725.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Pháo làm bùng nổ mạng xã hội"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-phu-nhan-co-con-yeu-cau-ngoc-kem-xin-loi-c51775308.epi" class="" title="ViruSs phủ nhận có con, yêu cầu Ngọc Kem xin lỗi" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51775308/76d0f767f12918774138.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51775308/76d0f767f12918774138.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_22_119_51775308/76d0f767f12918774138.jpg" alt="ViruSs phủ nhận có con, yêu cầu Ngọc Kem xin lỗi"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-phu-nhan-co-con-yeu-cau-ngoc-kem-xin-loi-c51775308.epi" class="" title="ViruSs phủ nhận có con, yêu cầu Ngọc Kem xin lỗi" target="_blank" rel="noopener noreferrer">ViruSs phủ nhận có con, yêu cầu Ngọc Kem xin lỗi</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/viruss-phu-nhan-co-con-yeu-cau-ngoc-kem-xin-loi-c51775308.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs phủ nhận có con, yêu cầu Ngọc Kem xin lỗi"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef] is-multi-photo flex-col group/photo"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ngoc-kem-livestream-chi-trich-thang-mat-viruss-hut-gan-200-000-mat-xem-c51774042.epi" class="" title="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem" target="_blank" rel="noopener noreferrer">Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem</a></h3></div><div class="bm-card-content m-0"><div class="h-full"><a href="/ngoc-kem-livestream-chi-trich-thang-mat-viruss-hut-gan-200-000-mat-xem-c51774042.epi" class="bm-multi-card-image relative flex justify-between w-full my-[5px] mx-0 h-[95px] rounded-[4px] overflow-hidden after:absolute after:left-[123px] after:bottom-[5px] after:w-[23px] after:h-[23px] after:rounded-[15px] after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] after:bg-[length:100px_4250px] after:bg-[0_-700px] after:bg-black/80 group-hover/photo:after:bg-[-50px_-700px]" title="" target="_blank" rel="noopener noreferrer"><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/7c83f468f2261b784237.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/7c83f468f2261b784237.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/7c83f468f2261b784237.jpg" alt="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/89037de87ba692f8cbb7.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/89037de87ba692f8cbb7.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/89037de87ba692f8cbb7.jpg" alt="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/2d32dcd9da9733c96a86.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/2d32dcd9da9733c96a86.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/2d32dcd9da9733c96a86.jpg" alt="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/bd6c428744c9ad97f4d8.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/bd6c428744c9ad97f4d8.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_21_180_51774042/bd6c428744c9ad97f4d8.jpg" alt="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem"></picture></figure></div></a></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tri-thuc-cuoc-song-tri-thuc-cuoc-song-p180.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tri thức &amp; Cuộc sống"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:64px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" alt="Báo Tri thức &amp; Cuộc sống" width="256" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/ngoc-kem-livestream-chi-trich-thang-mat-viruss-hut-gan-200-000-mat-xem-c51774042.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Ngọc Kem livestream chỉ trích thẳng mặt ViruSs, hút gần 200.000 mắt xem"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/rapper-phao-dang-trang-thai-an-y-giua-on-ao-ngoc-kem-viruss-c51774043.epi" class="" title="Rapper Pháo đăng trạng thái ẩn ý giữa ồn ào Ngọc Kem - ViruSs" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_180_51774043/4fddb436b2785b260269.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_180_51774043/4fddb436b2785b260269.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_180_51774043/4fddb436b2785b260269.jpg" alt="Rapper Pháo đăng trạng thái ẩn ý giữa ồn ào Ngọc Kem - ViruSs"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/rapper-phao-dang-trang-thai-an-y-giua-on-ao-ngoc-kem-viruss-c51774043.epi" class="" title="Rapper Pháo đăng trạng thái ẩn ý giữa ồn ào Ngọc Kem - ViruSs" target="_blank" rel="noopener noreferrer">Rapper Pháo đăng trạng thái ẩn ý giữa ồn ào Ngọc Kem - ViruSs</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tri-thuc-cuoc-song-tri-thuc-cuoc-song-p180.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tri thức &amp; Cuộc sống"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:64px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" alt="Báo Tri thức &amp; Cuộc sống" width="256" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/rapper-phao-dang-trang-thai-an-y-giua-on-ao-ngoc-kem-viruss-c51774043.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Rapper Pháo đăng trạng thái ẩn ý giữa ồn ào Ngọc Kem - ViruSs"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/phao-phan-hoi-thong-tin-hen-ho-viruss-c51774046.epi" class="" title="Pháo phản hồi thông tin hẹn hò ViruSs" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51774046/0b2cecc7ea8903d75a98.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51774046/0b2cecc7ea8903d75a98.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51774046/0b2cecc7ea8903d75a98.jpg" alt="Pháo phản hồi thông tin hẹn hò ViruSs"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/phao-phan-hoi-thong-tin-hen-ho-viruss-c51774046.epi" class="" title="Pháo phản hồi thông tin hẹn hò ViruSs" target="_blank" rel="noopener noreferrer">Pháo phản hồi thông tin hẹn hò ViruSs</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/phao-phan-hoi-thong-tin-hen-ho-viruss-c51774046.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Pháo phản hồi thông tin hẹn hò ViruSs"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/dong-thai-moi-nhat-cua-streamer-viruss-va-ngoc-kem-giua-lum-xum-tinh-cam-c51773656.epi" class="" title="Động thái mới nhất của streamer ViruSs và Ngọc Kem giữa lùm xùm tình cảm" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51773656/a42ff13af4741d2a4465.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51773656/a42ff13af4741d2a4465.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51773656/a42ff13af4741d2a4465.jpg" alt="Động thái mới nhất của streamer ViruSs và Ngọc Kem giữa lùm xùm tình cảm"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/dong-thai-moi-nhat-cua-streamer-viruss-va-ngoc-kem-giua-lum-xum-tinh-cam-c51773656.epi" class="" title="Động thái mới nhất của streamer ViruSs và Ngọc Kem giữa lùm xùm tình cảm" target="_blank" rel="noopener noreferrer">Động thái mới nhất của streamer ViruSs và Ngọc Kem giữa lùm xùm tình cảm</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:27px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/dong-thai-moi-nhat-cua-streamer-viruss-va-ngoc-kem-giua-lum-xum-tinh-cam-c51773656.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Động thái mới nhất của streamer ViruSs và Ngọc Kem giữa lùm xùm tình cảm"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-kiem-ca-tram-trieu-tu-buoi-livestream-dau-to-ban-gai-cu-c51772906.epi" class="" title="VirusS kiếm cả trăm triệu từ buổi livestream 'đấu tố' bạn gái cũ?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51772906/b640e342e60c0f52561d.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51772906/b640e342e60c0f52561d.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2_sm/2025_03_21_119_51772906/b640e342e60c0f52561d.jpg" alt="VirusS kiếm cả trăm triệu từ buổi livestream 'đấu tố' bạn gái cũ?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-kiem-ca-tram-trieu-tu-buoi-livestream-dau-to-ban-gai-cu-c51772906.epi" class="" title="VirusS kiếm cả trăm triệu từ buổi livestream 'đấu tố' bạn gái cũ?" target="_blank" rel="noopener noreferrer">VirusS kiếm cả trăm triệu từ buổi livestream 'đấu tố' bạn gái cũ?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/viruss-kiem-ca-tram-trieu-tu-buoi-livestream-dau-to-ban-gai-cu-c51772906.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="VirusS kiếm cả trăm triệu từ buổi livestream 'đấu tố' bạn gái cũ?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/ngoc-kem-la-ai-c51772910.epi" class="" title="Ngọc Kem là ai?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51772910/5f0e30ba37f4deaa87e5.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51772910/5f0e30ba37f4deaa87e5.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51772910/5f0e30ba37f4deaa87e5.jpg" alt="Ngọc Kem là ai?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ngoc-kem-la-ai-c51772910.epi" class="" title="Ngọc Kem là ai?" target="_blank" rel="noopener noreferrer">Ngọc Kem là ai?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/ngoc-kem-la-ai-c51772910.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Ngọc Kem là ai?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/giua-on-ao-viruss-va-nguoi-yeu-cu-mot-nu-ca-si-duoc-goi-ten-vi-tung-dan-mat-nam-streamer-c51772140.epi" class="" title="Giữa ồn ào ViruSs và người yêu cũ, một nữ ca sĩ được gọi tên vì từng 'dằn mặt' nam streamer?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_329_51772140/87edefc3ea8d03d35a9c.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_329_51772140/87edefc3ea8d03d35a9c.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_329_51772140/87edefc3ea8d03d35a9c.jpg" alt="Giữa ồn ào ViruSs và người yêu cũ, một nữ ca sĩ được gọi tên vì từng 'dằn mặt' nam streamer?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/giua-on-ao-viruss-va-nguoi-yeu-cu-mot-nu-ca-si-duoc-goi-ten-vi-tung-dan-mat-nam-streamer-c51772140.epi" class="" title="Giữa ồn ào ViruSs và người yêu cũ, một nữ ca sĩ được gọi tên vì từng 'dằn mặt' nam streamer?" target="_blank" rel="noopener noreferrer">Giữa ồn ào ViruSs và người yêu cũ, một nữ ca sĩ được gọi tên vì từng 'dằn mặt' nam streamer?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-saostar-saostar-p329.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí SaoStar"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:54px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" alt="Tạp chí SaoStar" width="216" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/giua-on-ao-viruss-va-nguoi-yeu-cu-mot-nu-ca-si-duoc-goi-ten-vi-tung-dan-mat-nam-streamer-c51772140.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Giữa ồn ào ViruSs và người yêu cũ, một nữ ca sĩ được gọi tên vì từng 'dằn mặt' nam streamer?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/tranh-luan-viruss-livestream-thu-tien-giua-on-ao-voi-ngoc-kem-c51771825.epi" class="" title="Tranh luận ViruSs livestream thu tiền giữa ồn ào với Ngọc Kem" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51771825/366ba220a76e4e30177f.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51771825/366ba220a76e4e30177f.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51771825/366ba220a76e4e30177f.jpg" alt="Tranh luận ViruSs livestream thu tiền giữa ồn ào với Ngọc Kem"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/tranh-luan-viruss-livestream-thu-tien-giua-on-ao-voi-ngoc-kem-c51771825.epi" class="" title="Tranh luận ViruSs livestream thu tiền giữa ồn ào với Ngọc Kem" target="_blank" rel="noopener noreferrer">Tranh luận ViruSs livestream thu tiền giữa ồn ào với Ngọc Kem</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:33px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/tranh-luan-viruss-livestream-thu-tien-giua-on-ao-voi-ngoc-kem-c51771825.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Tranh luận ViruSs livestream thu tiền giữa ồn ào với Ngọc Kem"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/rapper-phao-dang-story-day-an-y-bi-reo-ten-giua-on-ao-viruss-ngoc-kem-c51770040.epi" class="" title="Rapper Pháo đăng story đầy ẩn ý, bị 'réo tên' giữa ồn ào ViruSs - Ngọc Kem" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51770040/138d2f982ad6c3889ac7.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51770040/138d2f982ad6c3889ac7.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51770040/138d2f982ad6c3889ac7.jpg" alt="Rapper Pháo đăng story đầy ẩn ý, bị 'réo tên' giữa ồn ào ViruSs - Ngọc Kem"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/rapper-phao-dang-story-day-an-y-bi-reo-ten-giua-on-ao-viruss-ngoc-kem-c51770040.epi" class="" title="Rapper Pháo đăng story đầy ẩn ý, bị 'réo tên' giữa ồn ào ViruSs - Ngọc Kem" target="_blank" rel="noopener noreferrer">Rapper Pháo đăng story đầy ẩn ý, bị 'réo tên' giữa ồn ào ViruSs - Ngọc Kem</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" style="width:27px;height:16px" aria-label="Logo nhà xuất bản"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36<!-- --> liên quan</a><a href="/rapper-phao-dang-story-day-an-y-bi-reo-ten-giua-on-ao-viruss-ngoc-kem-c51770040.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Rapper Pháo đăng story đầy ẩn ý, bị 'réo tên' giữa ồn ào ViruSs - Ngọc Kem"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/toan-canh-vu-ngoc-kem-livestream-to-ban-trai-cu-ngoai-tinh-c51768891.epi" class="" title="Toàn cảnh vụ Ngọc Kem livestream tố bạn trai cũ ngoại tình" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51768891/d721797a7c34956acc25.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51768891/d721797a7c34956acc25.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_119_51768891/d721797a7c34956acc25.jpg" alt="Toàn cảnh vụ Ngọc Kem livestream tố bạn trai cũ ngoại tình"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/toan-canh-vu-ngoc-kem-livestream-to-ban-trai-cu-ngoai-tinh-c51768891.epi" class="" title="Toàn cảnh vụ Ngọc Kem livestream tố bạn trai cũ ngoại tình" target="_blank" rel="noopener noreferrer">Toàn cảnh vụ Ngọc Kem livestream tố bạn trai cũ ngoại tình</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/toan-canh-vu-ngoc-kem-livestream-to-ban-trai-cu-ngoai-tinh-c51768891.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Toàn cảnh vụ Ngọc Kem livestream tố bạn trai cũ ngoại tình"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/ban-gai-cu-livestream-to-viruss-ngoai-tinh-thu-hut-160-000-nguoi-xem-c51768698.epi" class="" title="Bạn gái cũ livestream tố ViruSs ngoại tình, thu hút 160.000 người xem" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_83_51768698/6bd67c0679489016c959.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_83_51768698/6bd67c0679489016c959.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_83_51768698/6bd67c0679489016c959.jpg" alt="Bạn gái cũ livestream tố ViruSs ngoại tình, thu hút 160.000 người xem"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ban-gai-cu-livestream-to-viruss-ngoai-tinh-thu-hut-160-000-nguoi-xem-c51768698.epi" class="" title="Bạn gái cũ livestream tố ViruSs ngoại tình, thu hút 160.000 người xem" target="_blank" rel="noopener noreferrer">Bạn gái cũ livestream tố ViruSs ngoại tình, thu hút 160.000 người xem</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-vtc-news-vtc-p83.epi" class="bm-card-source flex items-center shrink-0" title="Báo VTC News"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 62px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/6eede58338c0d19e88d1.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/6eede58338c0d19e88d1.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/6eede58338c0d19e88d1.png" alt="Báo VTC News" width="248" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/ban-gai-cu-livestream-to-viruss-ngoai-tinh-thu-hut-160-000-nguoi-xem-c51768698.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Bạn gái cũ livestream tố ViruSs ngoại tình, thu hút 160.000 người xem"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/toan-canh-vu-livestream-gan-200-000-nguoi-xem-ngoc-kem-to-viruss-ngoai-tinh-c51768709.epi" class="" title="Toàn cảnh vụ livestream gần 200.000 người xem Ngọc Kem tố ViruSs ngoại tình" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_20_51768709/2b2e78fe7db094eecda1.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_20_51768709/2b2e78fe7db094eecda1.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_20_51768709/2b2e78fe7db094eecda1.jpg" alt="Toàn cảnh vụ livestream gần 200.000 người xem Ngọc Kem tố ViruSs ngoại tình"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/toan-canh-vu-livestream-gan-200-000-nguoi-xem-ngoc-kem-to-viruss-ngoai-tinh-c51768709.epi" class="" title="Toàn cảnh vụ livestream gần 200.000 người xem Ngọc Kem tố ViruSs ngoại tình" target="_blank" rel="noopener noreferrer">Toàn cảnh vụ livestream gần 200.000 người xem Ngọc Kem tố ViruSs ngoại tình</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tien-phong-tien-phong-p20.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 50px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" alt="Báo Tiền Phong" width="200" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/toan-canh-vu-livestream-gan-200-000-nguoi-xem-ngoc-kem-to-viruss-ngoai-tinh-c51768709.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Toàn cảnh vụ livestream gần 200.000 người xem Ngọc Kem tố ViruSs ngoại tình"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-che-ngoc-kem-xau-tinh-nu-tiktoker-phan-phao-dung-co-bien-minh-thanh-nan-nhan-c51768539.epi" class="" title="ViruSs chê Ngọc Kem xấu tính, nữ TikToker phản pháo: 'Đừng cố biến mình thành nạn nhân'" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51768539/47aeeb1eee50070e5e41.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51768539/47aeeb1eee50070e5e41.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_21_105_51768539/47aeeb1eee50070e5e41.jpg" alt="ViruSs chê Ngọc Kem xấu tính, nữ TikToker phản pháo: 'Đừng cố biến mình thành nạn nhân'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-che-ngoc-kem-xau-tinh-nu-tiktoker-phan-phao-dung-co-bien-minh-thanh-nan-nhan-c51768539.epi" class="" title="ViruSs chê Ngọc Kem xấu tính, nữ TikToker phản pháo: 'Đừng cố biến mình thành nạn nhân'" target="_blank" rel="noopener noreferrer">ViruSs chê Ngọc Kem xấu tính, nữ TikToker phản pháo: 'Đừng cố biến mình thành nạn nhân'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 27px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/viruss-che-ngoc-kem-xau-tinh-nu-tiktoker-phan-phao-dung-co-bien-minh-thanh-nan-nhan-c51768539.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs chê Ngọc Kem xấu tính, nữ TikToker phản pháo: 'Đừng cố biến mình thành nạn nhân'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/gan-200-000-nguoi-xem-ngoc-kem-chi-trich-ban-trai-cu-c51765738.epi" class="" title="Gần 200.000 người xem Ngọc Kem chỉ trích bạn trai cũ" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_20_119_51765738/162f244b2005c95b9014.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_20_119_51765738/162f244b2005c95b9014.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_20_119_51765738/162f244b2005c95b9014.jpg" alt="Gần 200.000 người xem Ngọc Kem chỉ trích bạn trai cũ"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/gan-200-000-nguoi-xem-ngoc-kem-chi-trich-ban-trai-cu-c51765738.epi" class="" title="Gần 200.000 người xem Ngọc Kem chỉ trích bạn trai cũ" target="_blank" rel="noopener noreferrer">Gần 200.000 người xem Ngọc Kem chỉ trích bạn trai cũ</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/gan-200-000-nguoi-xem-ngoc-kem-chi-trich-ban-trai-cu-c51765738.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Gần 200.000 người xem Ngọc Kem chỉ trích bạn trai cũ"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/showbiz-19-3-nhan-sac-hoa-hau-dang-thu-thao-o-tuoi-34-c51755649.epi" class="" title="Showbiz 19/3: Nhan sắc Hoa hậu Đặng Thu Thảo ở tuổi 34" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_20_51755649/0f81698363cd8a93d3dc.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_20_51755649/0f81698363cd8a93d3dc.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_20_51755649/0f81698363cd8a93d3dc.jpg" alt="Showbiz 19/3: Nhan sắc Hoa hậu Đặng Thu Thảo ở tuổi 34"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/showbiz-19-3-nhan-sac-hoa-hau-dang-thu-thao-o-tuoi-34-c51755649.epi" class="" title="Showbiz 19/3: Nhan sắc Hoa hậu Đặng Thu Thảo ở tuổi 34" target="_blank" rel="noopener noreferrer">Showbiz 19/3: Nhan sắc Hoa hậu Đặng Thu Thảo ở tuổi 34</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tien-phong-tien-phong-p20.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 50px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/b3a88dc75084b9dae095.png" alt="Báo Tiền Phong" width="200" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/showbiz-19-3-nhan-sac-hoa-hau-dang-thu-thao-o-tuoi-34-c51755649.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Showbiz 19/3: Nhan sắc Hoa hậu Đặng Thu Thảo ở tuổi 34"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/tinh-duyen-trai-nguoc-cua-tu-hoang-streamer-c51751073.epi" class="" title="Tình duyên trái ngược của 'tứ hoàng streamer'" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51751073/b994e954e31a0a44530b.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51751073/b994e954e31a0a44530b.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51751073/b994e954e31a0a44530b.jpg" alt="Tình duyên trái ngược của 'tứ hoàng streamer'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/tinh-duyen-trai-nguoc-cua-tu-hoang-streamer-c51751073.epi" class="" title="Tình duyên trái ngược của 'tứ hoàng streamer'" target="_blank" rel="noopener noreferrer">Tình duyên trái ngược của 'tứ hoàng streamer'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/tinh-duyen-trai-nguoc-cua-tu-hoang-streamer-c51751073.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Tình duyên trái ngược của 'tứ hoàng streamer'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-xin-loi-thua-nhan-da-lua-doi-ngoc-kem-gui-loi-cam-on-den-ban-gai-cu-c51750500.epi" class="" title="ViruSs xin lỗi, thừa nhận đã lừa dối Ngọc Kem, gửi lời cảm ơn đến bạn gái cũ" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_105_51750500/ccc0c231c87f2121786e.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_105_51750500/ccc0c231c87f2121786e.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_105_51750500/ccc0c231c87f2121786e.jpg" alt="ViruSs xin lỗi, thừa nhận đã lừa dối Ngọc Kem, gửi lời cảm ơn đến bạn gái cũ"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-xin-loi-thua-nhan-da-lua-doi-ngoc-kem-gui-loi-cam-on-den-ban-gai-cu-c51750500.epi" class="" title="ViruSs xin lỗi, thừa nhận đã lừa dối Ngọc Kem, gửi lời cảm ơn đến bạn gái cũ" target="_blank" rel="noopener noreferrer">ViruSs xin lỗi, thừa nhận đã lừa dối Ngọc Kem, gửi lời cảm ơn đến bạn gái cũ</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 27px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/viruss-xin-loi-thua-nhan-da-lua-doi-ngoc-kem-gui-loi-cam-on-den-ban-gai-cu-c51750500.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs xin lỗi, thừa nhận đã lừa dối Ngọc Kem, gửi lời cảm ơn đến bạn gái cũ"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-nhan-nhu-den-ngoc-kem-sau-chia-tay-de-mat-em-la-loi-cua-anh-c51750036.epi" class="" title="ViruSs nhắn nhủ đến Ngọc Kem sau chia tay: 'Để mất em là lỗi của anh'" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_329_51750036/945b066c0322ea7cb333.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_329_51750036/945b066c0322ea7cb333.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_329_51750036/945b066c0322ea7cb333.jpg" alt="ViruSs nhắn nhủ đến Ngọc Kem sau chia tay: 'Để mất em là lỗi của anh'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-nhan-nhu-den-ngoc-kem-sau-chia-tay-de-mat-em-la-loi-cua-anh-c51750036.epi" class="" title="ViruSs nhắn nhủ đến Ngọc Kem sau chia tay: 'Để mất em là lỗi của anh'" target="_blank" rel="noopener noreferrer">ViruSs nhắn nhủ đến Ngọc Kem sau chia tay: 'Để mất em là lỗi của anh'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-saostar-saostar-p329.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí SaoStar"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 54px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" alt="Tạp chí SaoStar" width="216" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/viruss-nhan-nhu-den-ngoc-kem-sau-chia-tay-de-mat-em-la-loi-cua-anh-c51750036.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs nhắn nhủ đến Ngọc Kem sau chia tay: 'Để mất em là lỗi của anh'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-xin-loi-thua-nhan-lua-doi-ngoc-kem-c51749362.epi" class="" title="ViruSs xin lỗi, thừa nhận lừa dối Ngọc Kem?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51749362/85a11db214fcfda2a4ed.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51749362/85a11db214fcfda2a4ed.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_119_51749362/85a11db214fcfda2a4ed.jpg" alt="ViruSs xin lỗi, thừa nhận lừa dối Ngọc Kem?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-xin-loi-thua-nhan-lua-doi-ngoc-kem-c51749362.epi" class="" title="ViruSs xin lỗi, thừa nhận lừa dối Ngọc Kem?" target="_blank" rel="noopener noreferrer">ViruSs xin lỗi, thừa nhận lừa dối Ngọc Kem?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/viruss-xin-loi-thua-nhan-lua-doi-ngoc-kem-c51749362.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs xin lỗi, thừa nhận lừa dối Ngọc Kem?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/hot-girl-ngoc-kem-noi-gi-ve-qua-khu-yeu-viruss-c51748757.epi" class="" title="Hot girl Ngọc Kem nói gì về quá khứ yêu ViruSs?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_180_51748757/ecc762c76b8982d7db98.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_180_51748757/ecc762c76b8982d7db98.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_19_180_51748757/ecc762c76b8982d7db98.jpg" alt="Hot girl Ngọc Kem nói gì về quá khứ yêu ViruSs?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/hot-girl-ngoc-kem-noi-gi-ve-qua-khu-yeu-viruss-c51748757.epi" class="" title="Hot girl Ngọc Kem nói gì về quá khứ yêu ViruSs?" target="_blank" rel="noopener noreferrer">Hot girl Ngọc Kem nói gì về quá khứ yêu ViruSs?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tri-thuc-cuoc-song-tri-thuc-cuoc-song-p180.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tri thức &amp; Cuộc sống"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 64px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" alt="Báo Tri thức &amp; Cuộc sống" width="256" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/hot-girl-ngoc-kem-noi-gi-ve-qua-khu-yeu-viruss-c51748757.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Hot girl Ngọc Kem nói gì về quá khứ yêu ViruSs?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef] is-multi-photo flex-col group/photo"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ngoc-kem-thua-nhan-da-chia-tay-an-y-viruss-bat-ca-hai-tay-c51746076.epi" class="" title="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay" target="_blank" rel="noopener noreferrer">Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay</a></h3></div><div class="bm-card-content m-0"><div class="h-full"><a href="/ngoc-kem-thua-nhan-da-chia-tay-an-y-viruss-bat-ca-hai-tay-c51746076.epi" class="bm-multi-card-image relative flex justify-between w-full my-[5px] mx-0 h-[95px] rounded-[4px] overflow-hidden after:absolute after:left-[123px] after:bottom-[5px] after:w-[23px] after:h-[23px] after:rounded-[15px] after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] after:bg-[length:100px_4250px] after:bg-[0_-700px] after:bg-black/80 group-hover/photo:after:bg-[-50px_-700px]" title="" target="_blank" rel="noopener noreferrer"><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/773f949e9dd0748e2dc1.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/773f949e9dd0748e2dc1.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/773f949e9dd0748e2dc1.jpg" alt="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/a0546af563bb8ae5d3aa.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/a0546af563bb8ae5d3aa.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/a0546af563bb8ae5d3aa.jpg" alt="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/bbdd757c7c32956ccc23.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/bbdd757c7c32956ccc23.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/bbdd757c7c32956ccc23.jpg" alt="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay"></picture></figure></div><div class="bm-card-image relative overflow-hidden shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover rounded-none mt-0 w-[155px] h-[100px]"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/1f8fd32eda60333e6a71.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/1f8fd32eda60333e6a71.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w200_r4x3_sm/2025_03_18_180_51746076/1f8fd32eda60333e6a71.jpg" alt="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay"></picture></figure></div></a></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/bao-tri-thuc-cuoc-song-tri-thuc-cuoc-song-p180.epi" class="bm-card-source flex items-center shrink-0" title="Báo Tri thức &amp; Cuộc sống"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 64px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/e9b048137951900fc940.png" alt="Báo Tri thức &amp; Cuộc sống" width="256" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/ngoc-kem-thua-nhan-da-chia-tay-an-y-viruss-bat-ca-hai-tay-c51746076.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Ngọc Kem thừa nhận đã chia tay, ẩn ý ViruSs bắt cá hai tay"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/viruss-ngoc-kem-chia-tay-trong-on-ao-noi-gi-giua-don-doan-co-nguoi-thu-ba-c51745145.epi" class="" title="ViruSs - Ngọc Kem chia tay trong ồn ào, nói gì giữa đồn đoán có người thứ ba?" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_105_51745145/ac6e90be99f070ae29e1.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_105_51745145/ac6e90be99f070ae29e1.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_105_51745145/ac6e90be99f070ae29e1.jpg" alt="ViruSs - Ngọc Kem chia tay trong ồn ào, nói gì giữa đồn đoán có người thứ ba?"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/viruss-ngoc-kem-chia-tay-trong-on-ao-noi-gi-giua-don-doan-co-nguoi-thu-ba-c51745145.epi" class="" title="ViruSs - Ngọc Kem chia tay trong ồn ào, nói gì giữa đồn đoán có người thứ ba?" target="_blank" rel="noopener noreferrer">ViruSs - Ngọc Kem chia tay trong ồn ào, nói gì giữa đồn đoán có người thứ ba?</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/chuyen-trang-hoa-hoc-tro-bao-tien-phong-hht-p105.epi" class="bm-card-source flex items-center shrink-0" title="Chuyên trang Hoa Học Trò - Báo Tiền Phong"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 27px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/ae372a59f71a1e44470b.png" alt="Chuyên trang Hoa Học Trò - Báo Tiền Phong" width="108" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/viruss-ngoc-kem-chia-tay-trong-on-ao-noi-gi-giua-don-doan-co-nguoi-thu-ba-c51745145.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="ViruSs - Ngọc Kem chia tay trong ồn ào, nói gì giữa đồn đoán có người thứ ba?"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover before:absolute before:right-[5px] before:bottom-[5px] before:w-[23px] before:h-[23px] before:rounded-[15px] before:inline-block before:bg-black/50 before:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png')] before:bg-[length:100px_4250px] before:z-[2] before:bg-[position:0_-700px] group-hover/card:before:bg-[position:-50px_-700px] mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/chuyen-tinh-ngan-ngui-cua-ngoc-kem-va-viruss-c51741172.epi" class="" title="Chuyện tình ngắn ngủi của Ngọc Kem và ViruSs" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51741172/a49dd2e3daad33f36abc.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51741172/a49dd2e3daad33f36abc.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51741172/a49dd2e3daad33f36abc.jpg" alt="Chuyện tình ngắn ngủi của Ngọc Kem và ViruSs"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/chuyen-tinh-ngan-ngui-cua-ngoc-kem-va-viruss-c51741172.epi" class="" title="Chuyện tình ngắn ngủi của Ngọc Kem và ViruSs" target="_blank" rel="noopener noreferrer">Chuyện tình ngắn ngủi của Ngọc Kem và ViruSs</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/chuyen-tinh-ngan-ngui-cua-ngoc-kem-va-viruss-c51741172.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Chuyện tình ngắn ngủi của Ngọc Kem và ViruSs"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/drama-sau-chia-tay-viruss-phan-ung-kho-ngo-khi-ngoc-kem-an-y-bi-cam-sung-c51740321.epi" class="" title="Drama sau chia tay: ViruSs phản ứng khó ngờ khi Ngọc Kem ẩn ý bị 'cắm sừng'" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_329_51740321/3d2144cd4283abddf292.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_329_51740321/3d2144cd4283abddf292.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_329_51740321/3d2144cd4283abddf292.jpg" alt="Drama sau chia tay: ViruSs phản ứng khó ngờ khi Ngọc Kem ẩn ý bị 'cắm sừng'"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/drama-sau-chia-tay-viruss-phan-ung-kho-ngo-khi-ngoc-kem-an-y-bi-cam-sung-c51740321.epi" class="" title="Drama sau chia tay: ViruSs phản ứng khó ngờ khi Ngọc Kem ẩn ý bị 'cắm sừng'" target="_blank" rel="noopener noreferrer">Drama sau chia tay: ViruSs phản ứng khó ngờ khi Ngọc Kem ẩn ý bị 'cắm sừng'</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-saostar-saostar-p329.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí SaoStar"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 54px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/42bc76d3ab9042ce1b81.png" alt="Tạp chí SaoStar" width="216" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/drama-sau-chia-tay-viruss-phan-ung-kho-ngo-khi-ngoc-kem-an-y-bi-cam-sung-c51740321.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Drama sau chia tay: ViruSs phản ứng khó ngờ khi Ngọc Kem ẩn ý bị 'cắm sừng'"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div><div class="group/card bm-card relative max-w-full flex w-full [&amp;:not(:first-child)]:mt-[15px] [&amp;:not(:first-child)]:pt-[11px] [&amp;:not(:first-child)]:border-t [&amp;:not(:first-child)]:border-solid [&amp;:not(:first-child)]:border-[#e9ecef]"><div class="bm-card-image relative overflow-hidden rounded-[4px] shrink-0 after:absolute after:top-0 after:left-0 after:z-[-1] after:w-full after:h-full after:bg-[url('https://baomoi-static.bmcdn.me/web/styles/img/logo-baomoi-gray.png')] after:bg-[center_calc(50%-3px)] after:bg-[length:20%] after:bg-no-repeat after:opacity-40 [&amp;_figure]:h-full [&amp;_figure]:w-full [&amp;_figure]:overflow-hidden [&amp;_img:hover]:scale-[1.05] [&amp;_img]:transition-transform [&amp;_img]:duration-700 [&amp;_img]:object-cover [&amp;_video]:h-full [&amp;_video]:w-full [&amp;_video]:object-cover mt-[4px] w-[155px] h-[100px]"><div class="h-full"><a href="/ngoc-kem-va-viruss-chia-tay-c51738115.epi" class="" title="Ngọc Kem và ViruSs chia tay" target="_blank" rel="noopener noreferrer"><figure class="image lazy-image overflow-hidden opacity-0 !opacity-100 transition-opacity duration-200"><picture><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51738115/a4c212dd1d93f4cdad82.jpg.avif" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51738115/a4c212dd1d93f4cdad82.jpg.webp" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/w250_r3x2/2025_03_18_119_51738115/a4c212dd1d93f4cdad82.jpg" alt="Ngọc Kem và ViruSs chia tay"></picture></figure></a></div></div><div class="bm-card-content ml-[15px]"><div class="bm-card-header [&amp;_a]:group-hover/card:text-[var(--primary)] text-[2.2rem] leading-[2.8rem]"><h3 class="font-semibold block"><a href="/ngoc-kem-va-viruss-chia-tay-c51738115.epi" class="" title="Ngọc Kem và ViruSs chia tay" target="_blank" rel="noopener noreferrer">Ngọc Kem và ViruSs chia tay</a></h3></div><div class="bm-card-footer flex flex-wrap items-center text-[1.3rem] text-[#adb5bd] h-[20px] overflow-hidden [&amp;>:not(:first-child)]:ml-[12px] [&amp;>:not(:first-child)]:shrink-0"><a href="/tap-chi-tri-thuc-znews-p119.epi" class="bm-card-source flex items-center shrink-0" title="Tạp chí Tri thức"><figure class="image lazy-image overflow-hidden opacity-80 w-auto h-[16px] opacity-0 !opacity-100 transition-opacity duration-200" aria-label="Logo nhà xuất bản" style="width: 33px; height: 16px;"><picture><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/avif"><source srcset="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" type="image/webp"><img src="https://photo-baomoi.bmcdn.me/cb04900a3f46d6188f57.png" alt="Tạp chí Tri thức" width="132" height="64"></picture></figure></a><a href="/ngoc-kem-t39406623.epi" class="related inline !text-[#adb5bd] hover:!text-[var(--primary)]">36 liên quan</a><a href="/ngoc-kem-va-viruss-chia-tay-c51738115.epi" class="icon-detail ml-[10px] hidden leading-[0] group-hover/card:block [&amp;>i]:hover:bg-[position:-50px_0]" title="Ngọc Kem và ViruSs chia tay"><i class="bm-icon inline-block bg-[url(https://baomoi-static.bmcdn.me/web/styles/img/bm-icon-2.0.3.png)] bg-[length:100px_4250px] align-middle bg-[0_0] w-[20px] h-[20px]"></i></a></div></div></div></div>"""
soup = BeautifulSoup(html, "html.parser")

a_tags = soup.select("h3 > a")
urls = [f"https://baomoi.com{a.get('href')}" for a in a_tags]

In [28]:
import requests

for url in urls:
    res = requests.get(url)
    soup = BeautifulSoup(res.text, "html.parser")
    print(soup.select_one(".content-main time").get("datetime"))

2025-03-23T01:30:00.000Z
2025-03-22T12:36:00.000Z
2025-03-22T11:41:00.000Z
2025-03-22T09:46:24.000Z
2025-03-22T08:52:00.000Z
2025-03-22T03:08:02.000Z
2025-03-22T02:26:00.000Z
2025-03-21T18:04:39.000Z
2025-03-21T13:30:00.000Z
2025-03-21T13:30:00.000Z
2025-03-21T12:46:24.000Z
2025-03-21T12:28:43.000Z
2025-03-21T11:04:16.000Z
2025-03-21T10:15:41.000Z
2025-03-21T09:50:00.000Z
2025-03-21T09:21:02.000Z
2025-03-21T06:07:45.000Z
2025-03-21T04:19:13.000Z
2025-03-21T04:08:14.000Z
2025-03-21T04:05:00.000Z
2025-03-21T03:46:55.000Z
2025-03-20T15:55:05.000Z
2025-03-19T13:48:00.000Z
2025-03-19T05:29:59.000Z
2025-03-19T04:26:51.000Z
2025-03-19T04:01:00.000Z
2025-03-19T02:43:14.000Z


AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
2025-03-23
2025-03-22
2025-03-22
2025-03-22
2025-03-22
2025-03-22
2025-03-22
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-21
2025-03-20
2025-03-19
2025-03-19
2025-03-19
2025-03-19
2025-03-19